In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
USE_MANUAL_STORAGE_OVERRIDES = snakemake.params.manual_storage_overrides_mwh

In [ ]:
manual_storage_overrides_mwh = {
    'AT' : 3200000,
    'BE' : 22070.6,
    'BG' : 4000000,
    'CH' : 8400000,
    'CZ' : 1500000,
    'DE' : 300000,
    'DK' : 0,
    'EE' : 0,
    'ES' : 18400000,
    'FI' : 5500000,
    'FR' : 9800000,
    'GR' : 2300000,
    'HR' : 2800000,
    'HU' : 100000,
    'IE' : 1733,
    'IT' : 7900000,
    'LT' : 175030.6,
    'LU' : 3996096.4,
    'LV' : 1531951.2,
    'NL' : 0,
    'NO' : 84349253,
    'PL' : 1600000,
    'PT' : 2600000,
    'RO' : 12100000,
    'SE' : 33800000,
    'SI' : 2200000,
    'SK' : 2200000,
    'UK' : 908079.7,
}

In [ ]:
gem_file = snakemake.input.gem_file
hydro_file = snakemake.input.jrchydro
euroshape = snakemake.input.euroshape

In [ ]:
gdf = (
    gpd.read_file(gem_file)
    .query("status=='operating'")
    .loc[:, ["Technology", "index", "capacity_MW"]]
)

In [ ]:
df = (
    gdf.groupby(["Technology", "index"], as_index=False)["capacity_MW"]
    .sum()
    .rename(columns={"index": "region"})
)

In [ ]:
europe = gpd.read_file(euroshape)

regions = (
    europe[["index", "CNTR", "geometry"]]
    .rename(columns={"index": "region"})
    .copy()
)

In [ ]:
jrc_hydro = pd.read_csv(hydro_file)

hydro = (
    jrc_hydro
    .dropna(subset=["lon", "lat", "installed_capacity_MW", "type"])
    .loc[lambda x: x["type"] != "HPHS"]
    .copy()
)

hydro_gdf = gpd.GeoDataFrame(
    hydro,
    geometry=gpd.points_from_xy(hydro["lon"], hydro["lat"]),
    crs="EPSG:4326"
)

In [ ]:
hydro_technology_map = {
    "HDAM": "hydro_res",  # reservoir / dam hydro
    "HROR": "hydro_ror",  # run-of-river hydro
}

In [ ]:
#regions = europe[["index", "CNTR", "geometry"]].copy()

hydro_with_region = gpd.sjoin(
    hydro_gdf,
    regions[["region", "geometry"]],
    how="left",
    predicate="within"
)

hydro_with_region["Technology"] = hydro_with_region["type"].map(
    hydro_technology_map
)

In [ ]:
energy_cap = (
    hydro_with_region
)

In [ ]:
hydro_capacity_by_type = (
    hydro_with_region
    .dropna(subset=["region", "Technology"])
    .groupby(["Technology", "type", "region"], as_index=False)
    .agg(capacity_MW=("installed_capacity_MW", "sum"))
)

In [ ]:
hydro_technologies = sorted(set(hydro_technology_map.values()))

region_technology_grid = (
    regions[["region"]]
    .assign(_join_key=1)
    .merge(
        pd.DataFrame(
            {
                "Technology": hydro_technologies,
                "_join_key": 1,
            }
        ),
        on="_join_key",
        how="inner",
    )
    .drop(columns="_join_key")
)

In [ ]:
hydro_capacity = (
    hydro_capacity_by_type
    .groupby(["Technology", "region"], as_index=False)
    .agg(capacity_MW=("capacity_MW", "sum"))
)

In [ ]:
hydro_for_append = region_technology_grid.merge(
    hydro_capacity,
    on=["Technology", "region"],
    how="left",
)

hydro_for_append["capacity_MW"] = (
    hydro_for_append["capacity_MW"]
    .fillna(0.0)
)

In [ ]:
#hydro_for_append = (
#    hydro_capacity
#)

columns = ["Technology", "region", "capacity_MW"]

df = pd.concat(
    [
        df[columns],
        hydro_for_append[columns]
    ],
    ignore_index=True
)

df

In [ ]:
df = (
    df.groupby(["Technology", "region"], as_index=False)["capacity_MW"]
    .sum()
)

In [ ]:
technology_names = {
    "nuclear": "NuclearEPR",
    "solar": "Solar",
    "onshore": "Windonshore",
    "offshore": "Windoffshore",
    "hydro_ror": "HydroRoR",
    "hydro_res": "HydroRes",
}

df["Technology"] = df["Technology"].replace(technology_names)

In [ ]:
out = (
    df.pivot_table(
        index="Technology",
        columns="region",
        values="capacity_MW",
        aggfunc="sum",
    )
    .reset_index()
)

# Add the fixed columns shown in your target format
out.insert(1, "Year", 2050)
out.insert(2, "parameter", "gen_exist_pcap_z")

# Set limtype by technology -- adapt this dictionary if needed
limtype_map = {
    "HydroRoR": "UP",
    "HydroRes": "FX",
    "NuclearEPR": "FX",
    "Windonshore": "FX",
    "Windoffshore": "FX",
    "Solar": "FX",
}

out.insert(3, "limtype", out["Technology"].map(limtype_map))

# Optional: order region columns alphabetically
fixed_columns = ["Technology", "Year", "parameter", "limtype"]
region_columns = sorted(c for c in out.columns if c not in fixed_columns)
out = out[fixed_columns + region_columns]

# Leave unavailable technology/region combinations empty in the CSV
#out.to_csv(snakemake.output.gen_exist_z_pcap, index=False, na_rep="")
# We need to append hydroRes ecap_z later on. 

In [ ]:
id_cols = ["Technology", "Year", "parameter", "limtype"]

# Every other column is currently a zone (AT, BE, BG, ..., UKN)
zone_cols = [col for col in out.columns if col not in id_cols]

# Convert wide format to long format
gen_exist_r = out.melt(
    id_vars=id_cols,
    value_vars=zone_cols,
    var_name="zone",
    value_name="value"
)


In [ ]:
(
    gen_exist_r
    .query("Technology == ['Windonshore','Windoffshore','Solar'] & parameter == 'gen_exist_pcap_z'")
    .replace({'gen_exist_pcap_z' : 'gen_exist_pcap_r'})
    .assign(region = lambda x : x.zone)
    .fillna(0)
    .to_csv(snakemake.output.gen_exist_r,index=False)
)

In [ ]:
df_plot = (
    df.groupby(["Technology", "region"], as_index=False)["capacity_MW"]
      .sum()
)

In [ ]:
technologies = sorted(df_plot["Technology"].unique())

fig, axes = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(20, 12),
    constrained_layout=True
)

axes = axes.flatten()

for ax, tech in zip(axes, technologies):

    # Select capacity data for this technology
    tech_df = (
        df_plot.loc[df_plot["Technology"] == tech, ["region", "capacity_MW"]]
        .copy()
    )

    # Merge capacities onto the polygon regions.
    # europe["index"] contains region codes such as BE, UKL, SE33, etc.
    map_gdf = europe.merge(
        tech_df,
        how="left",
        left_on="index",
        right_on="region"
    )

    # Regions without this technology should show zero capacity
    map_gdf["capacity_MW"] = map_gdf["capacity_MW"].fillna(0)

    # Plot boundaries/background for regions with zero capacity
    map_gdf.plot(
        ax=ax,
        color="lightgrey",
        edgecolor="white",
        linewidth=0.3
    )

    # Plot only non-zero capacities, so zero-capacity regions remain grey
    map_gdf.loc[map_gdf["capacity_MW"] > 0].plot(
        ax=ax,
        column="capacity_MW",
        cmap="YlOrRd",
        legend=True,
        legend_kwds={
            "label": "Installed capacity (MW)",
            "shrink": 0.65
        },
        edgecolor="white",
        linewidth=0.3
    )

    ax.set_title(tech, fontsize=14)
    ax.set_axis_off()

# Hide unused axes if fewer than six technologies exist
for ax in axes[len(technologies):]:
    ax.set_visible(False)

plt.show()


In [ ]:
technologies = [
    "NaturalgasCCGTwithCCSnewOT",
    "NaturalgasOCGTnew",
    "Biomass",
    "BiomassCCS",
    "Solar",
    "Windoffshore",
    "Windonshore",
    "HydroRoR",
    "HydroRes",
    "NuclearEPR",
    "SynCon",
]

limtype_map = {
    "NaturalgasCCGTwithCCSnewOT": "UP",
    "NaturalgasOCGTnew": "UP",
    "Biomass": "UP",
    "BiomassCCS": "UP",
    "Solar": "UP",
    "Windoffshore": "UP",
    "Windonshore": "UP",
    "HydroRoR": "UP",
    "HydroRes": "FX",
    "NuclearEPR": "UP",
    "SynCon": "UP",
}

nuclear_limits = {
    "AT": 0.0,
    #"BE": 0.0,
    #"CH": 0.0,
    "DE1" : 0.0,
    "DE2": 0.0,
    "DEB": 0.0,
    "DE7": 0.0,
    "DEG": 0.0,
    "DEA": 0.0,
    "DEE": 0.0,
    "DE9": 0.0,
    "DE5": 0.0,
    "DE6": 0.0,
    "DEC": 0.0,
    "DED": 0.0,
    "DE4": 0.0,
    "DE3": 0.0,
    "DE8": 0.0,
    "DEF": 0.0,
    "DK0": 0.0,
    #"ES": 0.0,
    "GR": 0.0,
    "IT": 0.0,
    "LU": 0.0,
    "PT": 0.0,
}

In [ ]:
regs = sorted(europe["index"].dropna().unique())

In [ ]:
hydro_limits = (
    df.loc[
        df["Technology"].isin(["HydroRoR", "HydroRes"]),
        ["Technology", "region", "capacity_MW"]
    ]
    .groupby(["Technology", "region"], as_index=False)["capacity_MW"]
    .sum()
)


In [ ]:
gen_lim_z = pd.DataFrame(
    "inf",
    index=technologies,
    columns=regs,
    dtype=object
)

for region, limit_mw in nuclear_limits.items():
    if region in gen_lim_z.columns:
        gen_lim_z.loc["NuclearEPR", region] = limit_mw
    else:
        print(f"Warning: nuclear region '{region}' is not in europe['index'].")

# ----------------------------------------------------------
# 3. Hydro: set capacity equal to existing capacity
# ----------------------------------------------------------
# Hydro is 0 MW in regions without existing hydro capacity
for tech in ["HydroRoR", "HydroRes"]:
    gen_lim_z.loc[tech, :] = 0.0

for _, row in hydro_limits.iterrows():
    tech = row["Technology"]
    region = row["region"]
    capacity = row["capacity_MW"]

    if region in gen_lim_z.columns:
        gen_lim_z.loc[tech, region] = capacity
    else:
        print(
            f"Warning: hydro region '{region}' is not in europe['index'] "
            f"for technology '{tech}'."
        )

# Convert technology index back to a regular column
gen_lim_z = gen_lim_z.reset_index().rename(
    columns={"index": "Technology"}
)

# Add required fixed columns
gen_lim_z.insert(1, "Year", 2050)
gen_lim_z.insert(2, "parameter", "gen_lim_pcap_z")
gen_lim_z.insert(
    3,
    "limtype",
    gen_lim_z["Technology"].map(limtype_map)
)

# Ensure the fixed columns precede regional columns
fixed_columns = ["Technology", "Year", "parameter", "limtype"]
gen_lim_z = gen_lim_z[fixed_columns + regs]

gen_lim_z.head(10)


In [ ]:
hydro_storage = (
    hydro_with_region
    .loc[
        (hydro_with_region["type"] == "HDAM")
        & hydro_with_region["region"].notna(),
        ["region", "storage_capacity_MWh"]
    ]
    .groupby("region", as_index=False)
    .agg(storage_capacity_MWh=("storage_capacity_MWh", "sum"))
)

hydro_storage["storage_capacity_MWh"] = (
    hydro_storage["storage_capacity_MWh"]
    .fillna(0)
)


In [ ]:
# Reservoir power capacity per model region, in MW.
# This is used ONLY to distribute a manually specified country-level
# storage-capacity value across that country's regions.
hydro_res_power = (
    hydro_with_region
    .loc[
        (hydro_with_region["type"] == "HDAM")
        & hydro_with_region["region"].notna(),
        ["region", "installed_capacity_MW"]
    ]
    .groupby("region", as_index=False)
    .agg(reservoir_power_MW=("installed_capacity_MW", "sum"))
)


In [ ]:
if USE_MANUAL_STORAGE_OVERRIDES:
    # Turn the regional JRC storage table into a Series for easy replacement
    storage_by_region = (
        hydro_storage
        .set_index("region")["storage_capacity_MWh"]
        .copy()
    )

    # Make sure every model region has an entry, including regions with zero storage
    storage_by_region = storage_by_region.reindex(regs, fill_value=0.0)

    # Reservoir MW capacity by region, also including zero-capacity regions
    power_by_region = (
        hydro_res_power
        .set_index("region")["reservoir_power_MW"]
        .reindex(regs, fill_value=0.0)
    )

    for country, country_storage_mwh in manual_storage_overrides_mwh.items():

        # All model zones belonging to this country.
        # CNTR is the country code, while index is the model region code.
        country_regions = (
            europe.loc[europe["CNTR"] == country, "index"]
            .dropna()
            .unique()
            .tolist()
        )

        if not country_regions:
            print(
                f"Warning: no model regions found for country {country!r}; "
                "manual storage override was skipped."
            )
            continue

        # Only use regions that are present as columns in the output file
        country_regions = [
            region for region in country_regions
            if region in storage_by_region.index
        ]

        if float(country_storage_mwh) == 0:
            storage_by_region.loc[country_regions] = 0.0
    
            print(
                f"{country}: assigned 0.0 MWh over "
                f"{len(country_regions)} model region(s): {country_regions}"
            )
            continue
        
        # Sum of HydroRes / HDAM power capacity in this country
        country_power_mw = power_by_region.loc[country_regions].sum()

        if country_power_mw <= 0:
            print(
                f"Warning: cannot distribute {country!r} override "
                f"({country_storage_mwh:,.1f} MWh), because it has no mapped "
                "HDAM reservoir power capacity. Keeping JRC-derived values."
            )
            continue
        # Regional allocation shares based on reservoir power capacity:
        # regional storage = country storage × regional MW / country MW
        shares = power_by_region.loc[country_regions] / country_power_mw

        # Replace—not add to—the JRC-derived regional values for this country
        storage_by_region.loc[country_regions] = (
            float(country_storage_mwh) * shares
        )

        print(
            f"{country}: assigned {country_storage_mwh:,.1f} MWh "
            f"across {len(country_regions)} region(s)."
        )

    # Convert back to the usual DataFrame structure
    hydro_storage = (
        storage_by_region
        .rename("storage_capacity_MWh")
        .reset_index()
        .rename(columns={"index": "region"})
    )


In [ ]:
hydro_ecap_row = pd.DataFrame(
    [{
        "Technology": "HydroRes",
        "Year": 2050,
        "parameter": "gen_lim_ecap_z",
        "limtype": "FX",
        **{region: 0.0 for region in regs}
    }]
)

In [ ]:
for _, row in hydro_storage.iterrows():
    region = row["region"]
    storage_mwh = row["storage_capacity_MWh"]

    if region in hydro_ecap_row.columns:
        hydro_ecap_row.loc[0, region] = storage_mwh
    else:
        print(f"Warning: storage region {region!r} is not among model regions.")

In [ ]:
hydro_ecap_row = hydro_ecap_row[gen_lim_z.columns]

gen_lim_z = pd.concat(
    [gen_lim_z, hydro_ecap_row],
    ignore_index=True
)

In [ ]:
gen_exist_z = pd.concat(
    [out, hydro_ecap_row.replace({'gen_lim_ecap_z' : 'gen_exist_ecap_z'})],
    ignore_index=True
)

In [ ]:
def round_numeric(value):
    if isinstance(value, (int, float)) and pd.notna(value):
        return round(value, 1)
    return value  # retains strings such as "inf"

region_columns = [col for col in gen_lim_z.columns if col in regs]

gen_lim_z[region_columns] = gen_lim_z[region_columns].map(round_numeric)

gen_lim_z.to_csv(snakemake.output.gen_lim_z, index=False, na_rep="")

region_columns = [col for col in gen_exist_z.columns if col in regs]

gen_exist_z[region_columns] = gen_exist_z[region_columns].map(round_numeric)

gen_exist_z.to_csv(snakemake.output.gen_exist_z, index=False, na_rep="")

In [ ]:
pumped_hydro_gdf = jrc_hydro.loc[
    jrc_hydro["type"] == "HPHS"
].dropna(
    subset=["lon", "lat", "installed_capacity_MW"]
).copy()

pumped_hydro_gdf = gpd.GeoDataFrame(
    pumped_hydro_gdf,
    geometry=gpd.points_from_xy(
        pumped_hydro_gdf["lon"],
        pumped_hydro_gdf["lat"]
    ),
    crs="EPSG:4326",
).to_crs(europe.crs)

In [ ]:
pumped_hydro_with_region = gpd.sjoin(
    pumped_hydro_gdf,
    europe[["index", "geometry"]],
    how="left",
    predicate="within",
).rename(columns={"index": "region"})


In [ ]:
pumped_hydro = pumped_hydro_with_region.loc[
    pumped_hydro_with_region["region"].notna(),
    ["region", "installed_capacity_MW", "storage_capacity_MWh"],
].copy()

pumped_hydro["installed_capacity_MW"] = (
    pumped_hydro["installed_capacity_MW"].fillna(0)
)

pumped_hydro["storage_capacity_MWh"] = (
    pumped_hydro["storage_capacity_MWh"].fillna(0)
)

pumped_hydro_by_region = (
    pumped_hydro
    .groupby("region", as_index=False)
    .agg(
        power_capacity_MW=("installed_capacity_MW", "sum"),
        energy_capacity_MWh=("storage_capacity_MWh", "sum"),
    )
)

pumped_hydro_by_region[
    ["power_capacity_MW", "energy_capacity_MWh"]
] = pumped_hydro_by_region[
    ["power_capacity_MW", "energy_capacity_MWh"]
].round(1)

display(pumped_hydro_by_region)


In [ ]:
def make_storage_row(
    technology,
    parameter,
    limtype,
    regional_values=None,
    default_value=0.0,
    year=2050,
):
    """
    Create one wide-format storage input row.

    regional_values must be a Series indexed by model-region code,
    or a dict of {region: value}.
    """
    row = {
        "Technology": technology,
        "Year": year,
        "parameter": parameter,
        "limtype": limtype,
        **{region: default_value for region in regs},
    }

    if regional_values is not None:
        for region, value in regional_values.items():
            if region in row:
                row[region] = value

    return row


In [ ]:
pumped_power = (
    pumped_hydro_by_region
    .set_index("region")["power_capacity_MW"]
)

pumped_energy = (
    pumped_hydro_by_region
    .set_index("region")["energy_capacity_MWh"]
)


In [ ]:
store_exist_z = pd.DataFrame([
    make_storage_row(
        technology="PumpedHydro",
        parameter="store_exist_pcap_z",
        limtype="FX",
        regional_values=pumped_power,
        default_value=0.0,
    ),
    make_storage_row(
        technology="PumpedHydro",
        parameter="store_exist_ecap_z",
        limtype="FX",
        regional_values=pumped_energy,
        default_value=0.0,
    ),
])

fixed_columns = ["Technology", "Year", "parameter", "limtype"]

store_exist_z = store_exist_z[fixed_columns + regs]

display(store_exist_z)


In [ ]:
storage_limit_specification = [
    {
        "Technology": "PumpedHydro",
        "parameter": "store_lim_pcap_z",
        "limtype": "FX",
        "default": 0.0,
        "regional_values": pumped_power,
    },
    {
        "Technology": "PumpedHydro",
        "parameter": "store_lim_ecap_z",
        "limtype": "FX",
        "default": 0.0,
        "regional_values": pumped_energy,
    },

    # Technologies with unlimited power and energy capacity
    {
        "Technology": "Li-ion-8",
        "parameter": "store_lim_pcap_z",
        "limtype": "UP",
        "default": "inf",
        "regional_values": None,
    },
    {
        "Technology": "Li-ion-8",
        "parameter": "store_lim_ecap_z",
        "limtype": "UP",
        "default": "inf",
        "regional_values": None,
    },
    {
        "Technology": "H2-Tanks-OCGT",
        "parameter": "store_lim_pcap_z",
        "limtype": "UP",
        "default": "inf",
        "regional_values": None,
    },

    {
        "Technology": "H2-Tanks-CCGT",
        "parameter": "store_lim_pcap_z",
        "limtype": "UP",
        "default": "inf",
        "regional_values": None,
    },
    # Technologies unavailable in all regions
    {
        "Technology": "H2-Salt-OCGT",
        "parameter": "store_lim_pcap_z",
        "limtype": "UP",
        "default": 0.0,
        "regional_values": None,
    },
    {
        "Technology": "H2-Salt-CCGT",
        "parameter": "store_lim_pcap_z",
        "limtype": "UP",
        "default": 0.0,
        "regional_values": None,
    },

]


In [ ]:
store_lim_z = pd.DataFrame([
    make_storage_row(
        technology=item["Technology"],
        parameter=item["parameter"],
        limtype=item["limtype"],
        regional_values=item["regional_values"],
        default_value=item["default"],
    )
    for item in storage_limit_specification
])

store_lim_z = store_lim_z[fixed_columns + regs]

display(store_lim_z)


In [ ]:
def round_numeric_or_keep_string(value):
    if isinstance(value, (int, float)) and pd.notna(value):
        return round(value, 1)
    return value

store_lim_z[regs] = store_lim_z[regs].map(round_numeric_or_keep_string)
store_exist_z[regs] = store_exist_z[regs].map(round_numeric_or_keep_string)


In [ ]:
store_lim_z.to_csv(
    snakemake.output.store_lim_z,
    index=False,
    na_rep=""
)

store_exist_z.to_csv(
    snakemake.output.store_exist_z,
    index=False,
    na_rep=""
)
